# 🌿 PlantCare AI — Training Notebook

Advanced Plant Disease Detection using Transfer Learning with MobileNetV2.

**Dataset:** New Plant Diseases Dataset (87,000+ images, 38 classes)
**Model:** MobileNetV2 (ImageNet pretrained)
**Target:** ~96.6% validation accuracy

## Activity 1.1 — Setting Up the Environment

In [ ]:
# Install dependencies (run once)
# !pip install tensorflow==2.13.0 pillow numpy opencv-python matplotlib scikit-learn

## Activity 1.1.2 — Importing Required Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## Activity 1.1.3 — Download and Extract Dataset

Download from Kaggle: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset

In [ ]:
# If using Kaggle API:
# !kaggle datasets download -d vipoooool/new-plant-diseases-dataset
# !unzip new-plant-diseases-dataset.zip -d ../data/

DATASET_PATH = '../data/PlantVillage'
IMAGE_SIZE   = (224, 224)
BATCH_SIZE   = 32
NUM_CLASSES  = 38

print(f'Dataset path: {DATASET_PATH}')
print(f'Exists: {os.path.exists(DATASET_PATH)}')

## Activity 1.2 — Data Exploration and Visualization

In [ ]:
# Activity 1.2.1 — Configure Dataset Paths
classes = sorted(os.listdir(DATASET_PATH)) if os.path.exists(DATASET_PATH) else []
print(f'Number of classes: {len(classes)}')
for c in classes:
    n = len(os.listdir(os.path.join(DATASET_PATH, c)))
    print(f'  {c}: {n} images')

In [ ]:
# Activity 1.2.2 — Visualize Sample Images
import cv2
from pathlib import Path

fig, axes = plt.subplots(4, 6, figsize=(18, 12))
axes = axes.flatten()

sample_classes = classes[:24] if len(classes) >= 24 else classes
for i, cls in enumerate(sample_classes):
    cls_path = os.path.join(DATASET_PATH, cls)
    img_files = os.listdir(cls_path)
    if img_files:
        img = cv2.imread(os.path.join(cls_path, img_files[0]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img)
        label = cls.replace('___', '\n').replace('_', ' ')
        axes[i].set_title(label, fontsize=7)
    axes[i].axis('off')

plt.suptitle('Sample Leaf Images from Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Activity 1.2.3 — Dataset Statistics
total = 0
healthy_count = 0
disease_count = 0

for cls in classes:
    n = len(os.listdir(os.path.join(DATASET_PATH, cls)))
    total += n
    if 'healthy' in cls.lower(): healthy_count += n
    else: disease_count += n

print(f'Total images    : {total:,}')
print(f'Healthy images  : {healthy_count:,} ({100*healthy_count/total:.1f}%)')
print(f'Diseased images : {disease_count:,} ({100*disease_count/total:.1f}%)')
print(f'Num classes     : {len(classes)}')

## Activity 1.3 — Data Preprocessing and Augmentation

In [ ]:
# Setup Data Generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    fill_mode='nearest',
    validation_split=0.2
)

val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = train_datagen.flow_from_directory(
    DATASET_PATH, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)
val_gen = val_datagen.flow_from_directory(
    DATASET_PATH, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation'
)

print(f'Train batches      : {len(train_gen)}')
print(f'Validation batches : {len(val_gen)}')

## Activity 2.1 — Understanding Transfer Learning with MobileNetV2

MobileNetV2 features:
- **Efficient Architecture**: Depthwise separable convolutions
- **Inverted Residuals**: Linear bottleneck layers
- **Lightweight**: ~3.4M parameters vs ResNet50's 25M
- **Pre-trained**: ImageNet weights (1,000 classes)

## Activity 2.2 — Load Pre-trained MobileNetV2

In [ ]:
base_model = MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False
print(f'Base model layers     : {len(base_model.layers)}')
print(f'Base model parameters : {base_model.count_params():,} (frozen)')

## Activity 2.3 — Configure Transfer Learning Strategy

In [ ]:
# Build full model
inputs  = Input(shape=(*IMAGE_SIZE, 3))
x       = base_model(inputs, training=False)
x       = GlobalAveragePooling2D()(x)
x       = Dropout(0.35)(x)
x       = Dense(256, activation='relu')(x)
x       = Dropout(0.25)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)
model.summary()

## Activity 3.1 — Compile the Model

In [ ]:
LEARNING_RATE = 1e-4

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')]
)

print(f'Optimizer   : Adam (lr={LEARNING_RATE})')
print(f'Loss        : Categorical Crossentropy')
print(f'Metrics     : accuracy, top5_accuracy')

## Activity 3.2 — Setup Training Callbacks

In [ ]:
MODEL_SAVE_PATH = '../saved_models/plantcare_model.h5'
os.makedirs('../saved_models', exist_ok=True)

callbacks = [
    ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
    EarlyStopping(
        monitor='val_loss', patience=6, restore_best_weights=True, verbose=1
    )
]
print('Callbacks configured: ModelCheckpoint, ReduceLROnPlateau, EarlyStopping')

## Activity 3.3 — Train the Model

In [ ]:
EPOCHS = 20

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

## Activity 3.4 — Visualize Training History

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history.get('accuracy', []),     label='train_accuracy')
plt.plot(history.history.get('val_accuracy', []), label='val_accuracy')
plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.legend(); plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history.get('loss', []),     label='train_loss')
plt.plot(history.history.get('val_loss', []), label='val_loss')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title('Loss')

plt.tight_layout()
plt.savefig('../training_history.png', dpi=150)
plt.show()

best_val_acc = max(history.history.get('val_accuracy', [0]))
print(f'\n✅ Best Validation Accuracy: {best_val_acc*100:.2f}%')

## Activity 4.1 — Evaluate Model Performance

In [ ]:
results = model.evaluate(val_gen, verbose=1)
print(f'\nVal Loss     : {results[0]:.4f}')
print(f'Val Accuracy : {results[1]*100:.2f}%')
print(f'Val Top-5    : {results[2]*100:.2f}%')

In [ ]:
# Confusion matrix (top 10 classes)
from sklearn.metrics import classification_report
import itertools

val_gen.reset()
y_pred_probs = model.predict(val_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_gen.classes

class_names = list(val_gen.class_indices.keys())
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

## Activity 4.2 — Save the Final Model

In [ ]:
model.save(MODEL_SAVE_PATH)
print(f'✅ Model saved to: {MODEL_SAVE_PATH}')
print(f'   File size: {os.path.getsize(MODEL_SAVE_PATH) / 1e6:.1f} MB')

## Activity 5.1 — Run Flask Backend

After saving the model, run the Flask app:

```bash
python app.py
```

Visit: http://localhost:5000